# 07 — Deep Learning Model: Feed-Forward Neural Network

**Workstream**: Modeling — Deep Learning  
**Owner**: Aurelia  
**Goal**: Train and evaluate a neural-network model on the same engineered feature table used by the logistic regression and XGBoost notebooks.

This notebook adds a deep learning benchmark to the modeling suite.

It:
- Loads `data/processed/features.parquet`
- Uses the same forward-window label and temporal split as notebooks 04 and 05
- Trains a feed-forward MLP neural network
- Calibrates predicted probabilities using the validation set
- Evaluates PR-AUC, ROC-AUC, precision@k, Brier score, and decile lift
- Saves metrics to `reports/metrics/`
- Saves a trained model artifact to `data/models/`

The purpose is to test whether deep learning adds value beyond the current tabular baselines. If the MLP does not beat XGBoost, that is still useful evidence that tree-based models are better suited to this feature set.

In [ ]:
import sys
import json
import inspect
from datetime import date
from pathlib import Path

_PROJECT_ROOT = Path.cwd().parent

if str(_PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT / "src"))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.utils.class_weight import compute_sample_weight

from foodsafety.config import MODELS_DIR, PROCESSED_DIR, RANDOM_STATE
from foodsafety.utils.time import temporal_split, summarize
from foodsafety.models.baseline import ALL_FEATURES, LABEL_COL
from foodsafety.models.evaluate import (
    evaluate,
    decile_lift_table,
    calibration_table,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 220)

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

In [ ]:
features_path = PROCESSED_DIR / "features.parquet"

if not features_path.exists():
    raise SystemExit(
        f"Missing {features_path}. Run notebooks 02_label_construction and 03_feature_engineering first."
    )

features = pd.read_parquet(features_path)
features["inspection_date"] = pd.to_datetime(features["inspection_date"])

for c in features.columns:
    if c.startswith("flag_kw_"):
        features[c] = features[c].astype("int8")

print(f"features: {len(features):,} rows × {features.shape[1]} columns")
print(f"date range: {features['inspection_date'].min().date()} → {features['inspection_date'].max().date()}")
print(f"label: {LABEL_COL}")
print(f"positive rate: {features[LABEL_COL].mean():.2%}")

In [ ]:
building_cols = [
    c for c in features.columns
    if "building" in c.lower() or "violation" in c.lower()
]

feature_building_cols = [c for c in building_cols if c in ALL_FEATURES]

print(f"Building/violation columns in full table: {len(building_cols)}")
print(building_cols)

print(f"\nBuilding/violation columns included in ALL_FEATURES: {len(feature_building_cols)}")
print(feature_building_cols)

In [ ]:
TRAIN_END = "2024-07-01"
VAL_END = "2025-07-01"

split = temporal_split(
    features,
    train_end=TRAIN_END,
    val_end=VAL_END,
)

for name, frame in [
    ("train", split.train),
    ("val", split.val),
    ("test", split.test),
]:
    s = summarize(frame, label_col=LABEL_COL)
    print(
        f"{name:<5} n={s.rows:>6,} "
        f"dates {s.date_min.date()} → {s.date_max.date()} "
        f"positive_rate={s.positive_rate:.2%}"
    )

In [ ]:
missing_features = [c for c in ALL_FEATURES if c not in features.columns]

if missing_features:
    raise ValueError(
        f"These ALL_FEATURES are missing from features.parquet: {missing_features}"
    )

X_train = split.train[ALL_FEATURES].copy()
y_train = split.train[LABEL_COL].astype(int)

X_val = split.val[ALL_FEATURES].copy()
y_val = split.val[LABEL_COL].astype(int)

X_test = split.test[ALL_FEATURES].copy()
y_test = split.test[LABEL_COL].astype(int)

categorical_features = [
    c for c in ALL_FEATURES
    if X_train[c].dtype == "object" or str(X_train[c].dtype).startswith("category")
]

numeric_features = [
    c for c in ALL_FEATURES
    if c not in categorical_features
]

print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")

print(f"\nNumeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")
print(categorical_features)

In [ ]:
def make_onehot_encoder():
    """
    sklearn changed `sparse` to `sparse_output` in newer versions.
    This helper keeps the notebook compatible across versions.
    """
    kwargs = {
        "handle_unknown": "ignore",
        "min_frequency": 25,
    }

    if "sparse_output" in inspect.signature(OneHotEncoder).parameters:
        kwargs["sparse_output"] = False
    else:
        kwargs["sparse"] = False

    return OneHotEncoder(**kwargs)


numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_onehot_encoder()),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)

mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    batch_size=256,
    learning_rate_init=1e-3,
    max_iter=100,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=10,
    random_state=RANDOM_STATE,
    verbose=True,
)

pipe = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("mlp", mlp),
    ]
)

pipe

In [ ]:
%%time

# MLPClassifier support for sample_weight depends on sklearn version.
# Try class-balanced sample weights first; fall back to unweighted fit if unsupported.
sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)

try:
    pipe.fit(X_train, y_train, mlp__sample_weight=sample_weight)
    fit_mode = "class-balanced sample weights"
except TypeError as e:
    print("Sample weights not supported by this sklearn MLPClassifier version.")
    print("Falling back to unweighted MLP fit.")
    pipe.fit(X_train, y_train)
    fit_mode = "unweighted"

print("\nTraining complete.")
print(f"Fit mode: {fit_mode}")
print(f"MLP iterations: {pipe.named_steps['mlp'].n_iter_}")
print(f"Final training loss: {pipe.named_steps['mlp'].loss_:.4f}")

In [ ]:
# Newer scikit-learn versions replaced cv="prefit" with FrozenEstimator.
# This calibrates the already-trained MLP pipeline using the validation set.

try:
    from sklearn.frozen import FrozenEstimator

    model = CalibratedClassifierCV(
        FrozenEstimator(pipe),
        method="isotonic",
    )

except ImportError:
    # Fallback for older scikit-learn versions
    model = CalibratedClassifierCV(
        pipe,
        method="isotonic",
        cv="prefit",
    )

model.fit(X_val, y_val)

print("Calibrated MLP model on validation set.")

In [ ]:
val_scores = model.predict_proba(X_val)[:, 1]
test_scores = model.predict_proba(X_test)[:, 1]

val_report = evaluate(y_val, val_scores)
test_report = evaluate(y_test, test_scores)

mlp_summary = pd.DataFrame(
    {
        "val": val_report.to_dict(),
        "test": test_report.to_dict(),
    }
)

mlp_summary.round(4)

In [ ]:
lift = decile_lift_table(y_test, test_scores)

print("Decile lift table — MLP test set")
print(lift.round(4).to_string())

In [ ]:
calib = calibration_table(y_test, test_scores, n_bins=10)

print("Calibration table — MLP test set")
print(calib.round(4).to_string())

fig, ax = plt.subplots(figsize=(5, 5))

ax.plot([0, 1], [0, 1], linestyle=":", label="Perfect calibration")
ax.plot(
    calib["mean_predicted"],
    calib["mean_observed"],
    marker="o",
    label="MLP neural network",
)

ax.set_xlabel("Mean predicted risk")
ax.set_ylabel("Mean observed risk")
ax.set_title("Calibration curve — MLP test set")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
metrics_dir = _PROJECT_ROOT / "reports" / "metrics"
metrics_dir.mkdir(parents=True, exist_ok=True)

comparison_rows = []

for path in sorted(metrics_dir.glob("*.json")):
    try:
        report = json.loads(path.read_text())

        if "test" not in report:
            continue

        row = {
            "model": report.get("model", path.stem),
            "file": path.name,
        }
        row.update(report["test"])
        comparison_rows.append(row)

    except Exception as e:
        print(f"Skipping {path.name}: {e}")

mlp_row = {
    "model": "mlp_neural_network",
    "file": "current_notebook",
}
mlp_row.update(test_report.to_dict())
comparison_rows.append(mlp_row)

comparison = pd.DataFrame(comparison_rows)

preferred_cols = [
    "model",
    "file",
    "pr_auc",
    "roc_auc",
    "precision_at_5pct",
    "precision_at_10pct",
    "precision_at_20pct",
    "top_decile_lift",
    "brier_score",
]

available_cols = [c for c in preferred_cols if c in comparison.columns]

comparison[available_cols].sort_values("pr_auc", ascending=False).round(4)

In [ ]:
existing = comparison[comparison["model"] != "mlp_neural_network"].copy()

if len(existing) == 0:
    print("No existing metrics found to compare against.")
else:
    best_existing = existing.sort_values("pr_auc", ascending=False).iloc[0]
    mlp_metrics = test_report.to_dict()

    print("Best existing model by PR-AUC:")

    display_cols = [c for c in available_cols if c in best_existing.index]
    best_existing_display = best_existing[display_cols].copy()

    # Round only numeric values, leave strings like model/file unchanged
    for col in best_existing_display.index:
        if isinstance(best_existing_display[col], (int, float, np.integer, np.floating)):
            best_existing_display[col] = round(best_existing_display[col], 4)

    print(best_existing_display.to_string())

    print("\nMLP test metrics:")
    print(pd.Series(mlp_metrics).round(4).to_string())

    print("\nDelta: MLP - best existing")
    for metric in [
        "pr_auc",
        "roc_auc",
        "precision_at_10pct",
        "top_decile_lift",
        "brier_score",
    ]:
        if metric in best_existing and metric in mlp_metrics:
            try:
                delta = float(mlp_metrics[metric]) - float(best_existing[metric])
                print(f"{metric:<20} {delta:+.4f}")
            except Exception:
                print(f"{metric:<20} could not compare")

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

reports_metrics_dir = _PROJECT_ROOT / "reports" / "metrics"
reports_metrics_dir.mkdir(parents=True, exist_ok=True)

stamp = date.today().isoformat().replace("-", "")

model_path = MODELS_DIR / f"mlp_{stamp}.joblib"
metadata_path = MODELS_DIR / f"mlp_{stamp}_metadata.json"
report_path = reports_metrics_dir / f"mlp_{stamp}.json"

joblib.dump(model, model_path)

metadata = {
    "model": "mlp_neural_network_isotonic",
    "random_state": RANDOM_STATE,
    "date_trained": date.today().isoformat(),
    "fit_mode": fit_mode,
    "split": {
        "train_end": str(split.train_end.date()),
        "val_end": str(split.val_end.date()),
        "train_n": int(len(split.train)),
        "val_n": int(len(split.val)),
        "test_n": int(len(split.test)),
    },
    "architecture": {
        "hidden_layer_sizes": [128, 64, 32],
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-4,
        "batch_size": 256,
        "learning_rate_init": 1e-3,
        "max_iter": 100,
        "early_stopping": True,
        "validation_fraction": 0.15,
        "n_iter_no_change": 10,
    },
    "features": {
        "all": ALL_FEATURES,
        "numeric": numeric_features,
        "categorical": categorical_features,
        "building_or_violation_features_in_table": building_cols,
        "building_or_violation_features_in_model": feature_building_cols,
        "label_col": LABEL_COL,
    },
    "metrics": {
        "val": val_report.to_dict(),
        "test": test_report.to_dict(),
    },
    "features_parquet_mtime": pd.Timestamp(
        features_path.stat().st_mtime,
        unit="s",
    ).isoformat(),
}

report = {
    "model": "mlp_neural_network",
    "date_trained": date.today().isoformat(),
    "fit_mode": fit_mode,
    "val": val_report.to_dict(),
    "test": test_report.to_dict(),
}

with metadata_path.open("w") as f:
    json.dump(metadata, f, indent=2)

with report_path.open("w") as f:
    json.dump(report, f, indent=2)

print(f"saved model    → {model_path}")
print(f"saved metadata → {metadata_path}")
print(f"saved metrics  → {report_path}")
print(f"model size: {model_path.stat().st_size / 1e6:.1f} MB")

## Handoff

This notebook adds a neural-network benchmark to the Food Safety Intelligence modeling suite.

### What was added

- A feed-forward MLP neural network trained on the same `features.parquet` table as the existing logistic regression and XGBoost models
- Same temporal train/validation/test split as notebooks 04 and 05
- Probability calibration using isotonic calibration on the validation split
- Evaluation using the same project metrics:
  - PR-AUC
  - ROC-AUC
  - precision@5%
  - precision@10%
  - precision@20%
  - top-decile lift
  - Brier score
- Saved metrics report under `reports/metrics/`
- Saved model artifact under `data/models/`

### How to interpret

If the MLP outperforms XGBoost on PR-AUC and precision@10%, it may be worth considering as an inference candidate.

If the MLP does not outperform XGBoost, the result is still useful: it shows that the current tabular feature set is likely better served by tree-based modeling than by a neural network.

### Notes for backend / app work

The trained model is saved as:

`data/models/mlp_<date>.joblib`

The comparable metrics report is saved as:

`reports/metrics/mlp_<date>.json`

The model can be loaded with `joblib.load(...)` and called with:

`model.predict_proba(feature_frame[ALL_FEATURES])[:, 1]`